2.2 — Mini-BPE learner 

In [1]:
from collections import Counter

corpus = [
    ("low", 4),
    ("lowest", 2),
    ("newer", 6),
    ("wider", 2),
    ("new", 2),
]

# Add end-of-word marker and split into characters
data = {
    tuple(list(word) + ["_"]): count
    for word, count in corpus
}

vocab = set()
for tokens in data:
    vocab.update(tokens)


def get_pairs(data):
    pairs = Counter()

    for tokens, freq in data.items():
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += freq

    return pairs


def merge_pair(data, pair):
    new_data = {}

    for tokens, freq in data.items():
        new_tokens = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                new_tokens.append(pair[0] + pair[1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        new_data[tuple(new_tokens)] = freq

    return new_data


num_merges = 8

for step in range(num_merges):
    pairs = get_pairs(data)

    if not pairs:
        break

    best_pair, count = pairs.most_common(1)[0]
    new_token = best_pair[0] + best_pair[1]

    data = merge_pair(data, best_pair)
    vocab.add(new_token)

    print(
        f"Step {step + 1}: "
        f"{best_pair} -> {new_token}, "
        f"count={count}, "
        f"vocab size={len(vocab)}"
    )


Step 1: ('w', 'e') -> we, count=8, vocab size=12
Step 2: ('n', 'e') -> ne, count=8, vocab size=13
Step 3: ('r', '_') -> r_, count=8, vocab size=14
Step 4: ('l', 'o') -> lo, count=6, vocab size=15
Step 5: ('w', '_') -> w_, count=6, vocab size=16
Step 6: ('ne', 'we') -> newe, count=6, vocab size=17
Step 7: ('newe', 'r_') -> newer_, count=6, vocab size=18
Step 8: ('lo', 'w_') -> low_, count=4, vocab size=19


2.3 — BPE on English language

In [4]:
from collections import Counter
import re

text = """
Machine learning models process language in many different ways.
Subword tokenization helps models understand common words and rare words.
A tokenizer can break unfamiliar words into smaller meaningful pieces.
These pieces can represent prefixes, stems, suffixes, or complete words.
This approach makes the vocabulary smaller and helps reduce the OOV problem.
"""

# Get words
words = re.findall(r"[A-Za-z]+", text.lower())

# Add end-of-word marker
vocab = Counter()

for word in words:
    vocab[tuple(list(word) + ["_"])] += 1


def get_pairs(vocab):
    pairs = Counter()

    for tokens, freq in vocab.items():
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += freq

    return pairs


def merge_pair(vocab, pair):
    new_vocab = Counter()

    for tokens, freq in vocab.items():
        new_tokens = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                new_tokens.append(pair[0] + pair[1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        new_vocab[tuple(new_tokens)] += freq

    return new_vocab


# Initial vocabulary
token_vocab = set()

for tokens in vocab:
    token_vocab.update(tokens)

merges = []

# Learn 30 merges
for step in range(30):
    pairs = get_pairs(vocab)

    if not pairs:
        break

    best_pair, count = pairs.most_common(1)[0]
    new_token = best_pair[0] + best_pair[1]

    merges.append((best_pair, new_token, count))

    vocab = merge_pair(vocab, best_pair)
    token_vocab.add(new_token)

    print(
        f"Merge {step + 1}: "
        f"{best_pair} -> {new_token}, "
        f"count = {count}, "
        f"vocab size = {len(token_vocab)}"
    )


Merge 1: ('s', '_') -> s_, count = 17, vocab size = 26
Merge 2: ('e', '_') -> e_, count = 8, vocab size = 27
Merge 3: ('a', 'n') -> an, count = 8, vocab size = 28
Merge 4: ('r', 'e') -> re, count = 6, vocab size = 29
Merge 5: ('o', 'r') -> or, count = 6, vocab size = 30
Merge 6: ('i', 'n') -> in, count = 5, vocab size = 31
Merge 7: ('l', 'e') -> le, count = 5, vocab size = 32
Merge 8: ('w', 'or') -> wor, count = 5, vocab size = 33
Merge 9: ('wor', 'd') -> word, count = 5, vocab size = 34
Merge 10: ('e', 's_') -> es_, count = 5, vocab size = 35
Merge 11: ('m', 'a') -> ma, count = 4, vocab size = 36
Merge 12: ('a', 'r') -> ar, count = 4, vocab size = 37
Merge 13: ('e', 'l') -> el, count = 4, vocab size = 38
Merge 14: ('word', 's_') -> words_, count = 4, vocab size = 39
Merge 15: ('t', 'h') -> th, count = 4, vocab size = 40
Merge 16: ('m', 'o') -> mo, count = 3, vocab size = 41
Merge 17: ('p', 'r') -> pr, count = 3, vocab size = 42
Merge 18: ('pr', 'o') -> pro, count = 3, vocab size = 43


Q5 Naive tokenization code

In [10]:
paragrpah = """Aaj main apne dost ke saath bazaar gaya.
Wahan humne kuchh phal aur sabziyaan khareedi, lekin baarish shuru ho gayi.
Isliye hum jaldi se ghar wapas aa gaye.
Ghar pahunchkar humne garam chai pi."""

naive_tokens = paragraph.split()
print("Naive space-based tokens:")
for token in naive_tokens:
    print(token)

Naive space-based tokens:
BPE
tokenization
helps
language
models
handle
new
words.
Common
words
and
common
word
parts
become
reusable
subword
tokens.
Rare
words
can
be
represented
by
combining
smaller
tokens.
This
makes
the
vocabulary
efficient
while
preserving
useful
patterns.
Subword
tokens
can
capture
stems,
prefixes,
and
suffixes.
